In [1]:
%pip install tabfm[pytorch]

  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/771.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/771.9 kB ? eta -:--:--
   ------------- -------------------------- 262.1/771.9 kB ? eta -:--:--
   --------------------------- ------------ 524.3/771.9 kB 1.2 MB/s eta 0:00:01
   ---------------------------------------- 771.9/771.9 kB 1.2 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.0 MB 577.6 kB/s eta 0:00:06
   ----- ---------------------------------- 0.5/4.0 MB 577.6 kB/s eta 0:00:06
   ------- -------------------------------- 0.8/4.0 MB 654.7 kB/s eta 0:00:05
   ------- -------------------------------- 0.8/4.0 MB 65


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
"""
TabFM (tabular foundation model) classifier on the Planning Relax dataset.
Evaluated the same rigorous way as the other models: LOO-CV, balanced_accuracy,
f1_macro, confusion matrix, and comparison against the dummy baseline.
"""

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, LeaveOneOut, cross_val_score
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    balanced_accuracy_score, f1_score, roc_auc_score
)

from tabfm import TabFMClassifier, tabfm_v1_0_0_pytorch as tabfm_v1_0_0

RANDOM_STATE = 42

# =====================================================================
# 1. Load data
# =====================================================================
df = pd.read_csv('plrx.txt', delimiter='\t', header=None)
X = df.iloc[:, :12].values
y = df.iloc[:, 12].values

print("Class distribution:", dict(zip(*np.unique(y, return_counts=True))))

# =====================================================================
# 2. Simple train/test split — quick sanity check first
# =====================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

model = tabfm_v1_0_0.load(model_type="classification")
clf = TabFMClassifier(model=model)
clf.fit(X_train, y_train)

probs = clf.predict_proba(X_test)          # shape (n_samples, n_classes)
preds = clf.predict(X_test)                # hard labels

print("\n" + "="*60)
print("HOLDOUT TEST SET RESULT")
print("="*60)
print(classification_report(y_test, preds))
print("Confusion matrix:")
print(confusion_matrix(y_test, preds))

bal_acc = balanced_accuracy_score(y_test, preds)
f1_macro = f1_score(y_test, preds, average='macro')
print(f"\nbalanced_accuracy: {bal_acc:.3f}")
print(f"f1_macro:          {f1_macro:.3f}")

# ROC-AUC using the probability of the minority/positive class (2.0)
try:
    classes_ = clf.classes_ if hasattr(clf, 'classes_') else np.unique(y_train)
    pos_idx = list(classes_).index(2.0)
    auc = roc_auc_score(y_test, probs[:, pos_idx])
    print(f"roc_auc:            {auc:.3f}")
except Exception as e:
    print(f"(Skipped ROC-AUC: {e})")

# =====================================================================
# 3. Leave-One-Out CV — the trustworthy estimate given N=182
# =====================================================================
# Note: TabFM is a pretrained foundation model, typically used zero-shot /
# few-shot per call rather than retrained from scratch like sklearn
# estimators. Refitting it fresh inside each of the 182 LOO folds mirrors
# how you'd use it in practice (in-context fit on the training fold, then
# predict the held-out point) and avoids any leakage.

print("\n" + "="*60)
print("LEAVE-ONE-OUT CV")
print("="*60)

loo = LeaveOneOut()
loo_preds, loo_true = [], []

for train_idx, test_idx in loo.split(X):
    X_tr, X_te = X[train_idx], X[test_idx]
    y_tr, y_te = y[train_idx], y[test_idx]

    model_loo = tabfm_v1_0_0.load(model_type="classification")
    clf_loo = TabFMClassifier(model=model_loo)
    clf_loo.fit(X_tr, y_tr)
    pred = clf_loo.predict(X_te)

    loo_preds.append(pred[0])
    loo_true.append(y_te[0])

loo_preds = np.array(loo_preds)
loo_true = np.array(loo_true)

print("\nLOO classification report:")
print(classification_report(loo_true, loo_preds))
print("LOO confusion matrix:")
print(confusion_matrix(loo_true, loo_preds))
print(f"\nLOO accuracy:          {(loo_preds == loo_true).mean():.3f}")
print(f"LOO balanced_accuracy: {balanced_accuracy_score(loo_true, loo_preds):.3f}")
print(f"LOO f1_macro:          {f1_score(loo_true, loo_preds, average='macro'):.3f}")

# =====================================================================
# 4. Compare against dummy baseline (LOO)
# =====================================================================
dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
dummy_loo_scores = cross_val_score(dummy, X, y, cv=loo, scoring='accuracy')
print(f"\nDummy (most_frequent) LOO accuracy: {dummy_loo_scores.mean():.3f}")
print(f"TabFM LOO accuracy:                 {(loo_preds == loo_true).mean():.3f}")

Class distribution: {np.float64(1.0): np.int64(130), np.float64(2.0): np.int64(52)}


c:\Users\akila\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]